# FICOS — Walk-Forward Directional Confusion Matrix Analysis
This notebook computes directional confusion matrices (Actual UP/DOWN vs Predicted UP/DOWN)  
across all 5 walk-forward folds for all promoted pairs (and baseline pairs).

- **Ungated**: Directional accuracy across all out-of-sample test days.
- **Gated**: Directional accuracy when model signals clear the P10/P90 empirical residual uncertainty gate.


In [ ]:
import os
REPO = "https://github.com/SSOHEB/FICOS-Platform.git"
if not os.path.exists("FICOS-Platform"):
    !git clone {REPO}
os.chdir("FICOS-Platform")
print("Working dir:", os.getcwd())


In [ ]:
!pip install -q lightgbm xgboost scikit-learn pandas numpy matplotlib seaborn
print("Packages installed.")


In [ ]:
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import confusion_matrix
import xgboost as xgb, lightgbm as lgb

warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('outputs/modeling_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
n_rows = len(df)

all_cols     = list(df.columns)
leakage_cols = [c for c in all_cols if c.startswith('dir_') or
                c.startswith('future_') or c.startswith('target_')]
drop_cols    = set(leakage_cols + ['date'])
feature_cols = [c for c in all_cols if c not in drop_cols]

test_size, val_size = 250, 200
fold_configs = {}
for f in range(1, 6):
    te_end   = n_rows - (5 - f) * test_size
    te_start = te_end  - test_size
    va_end, va_start = te_start, te_start - val_size
    fold_configs[f] = dict(tr_end=va_start, va_start=va_start,
                           va_end=va_end, te_start=te_start, te_end=te_end)

EVAL_PAIRS = [
    ('cape', 1),
    ('panamax', 1),
    ('supramax', 1),
    ('supramax', 7),
    ('handy', 1),
    ('handy', 7),
    ('supramax', 14), # Baseline comparison
    ('kdci', 7),       # Baseline comparison
]

print(f"Dataset: {n_rows} rows | {len(feature_cols)} clean features")
print(f"Evaluating {len(EVAL_PAIRS)} asset-horizon pairs across 5 folds...")


In [ ]:
def smape(a, b):
    return float(np.mean(200*np.abs(b-a)/(np.abs(a)+np.abs(b)+1e-8)))

all_pair_records = {}

for (asset, h) in EVAL_PAIRS:
    pair_key = f"{asset}_{h}d"
    print(f"Processing {pair_key}...", flush=True)
    records = []

    for fold_id in range(1, 6):
        cfg = fold_configs[fold_id]
        tr_end, va_start, va_end, te_start, te_end = (
            cfg['tr_end'], cfg['va_start'], cfg['va_end'], cfg['te_start'], cfg['te_end'])

        df_p       = df.copy()
        df_p['_yd'] = df_p[asset].shift(-h) - df_p[asset]
        df_v       = df_p[~df_p['_yd'].isna()].reset_index(drop=True)

        tr_m = np.zeros(len(df_v), bool); tr_m[:tr_end]         = True
        va_m = np.zeros(len(df_v), bool); va_m[va_start:va_end] = True
        te_m = np.zeros(len(df_v), bool); te_m[te_start:te_end] = True

        X_raw  = df_v[feature_cols].values.copy()
        y_d    = df_v['_yd'].values
        y_base = df_v[asset].values

        med = np.nanmedian(X_raw[tr_m], axis=0); med[np.isnan(med)] = 0.0
        for ci in range(X_raw.shape[1]):
            X_raw[:, ci] = np.where(np.isnan(X_raw[:, ci]), med[ci], X_raw[:, ci])

        sx  = StandardScaler()
        Xtr = sx.fit_transform(X_raw[tr_m])
        Xva = sx.transform(X_raw[va_m])
        Xte = sx.transform(X_raw[te_m])

        sy     = StandardScaler()
        ytr_sc = sy.fit_transform(y_d[tr_m].reshape(-1,1)).flatten()

        sel   = SelectKBest(f_regression, k=30)
        Xtr_s = sel.fit_transform(Xtr, y_d[tr_m])
        Xva_s = sel.transform(Xva)
        Xte_s = sel.transform(Xte)

        models = {}

        # Ridge
        br, brv = None, float('inf')
        for a in [0.1, 1.0, 10.0, 100.0, 1000.0]:
            m = Ridge(alpha=a).fit(Xtr_s, y_d[tr_m])
            s = smape(y_d[va_m], m.predict(Xva_s))
            if s < brv: brv, br = s, m
        models['Ridge'] = (br, br.predict(Xva_s), br.predict(Xte_s))

        # ElasticNet
        be, bev, en_va, en_te = None, float('inf'), None, None
        for a in [0.01, 0.1, 1.0]:
            for l1 in [0.2, 0.5, 0.8]:
                m    = ElasticNet(alpha=a, l1_ratio=l1, max_iter=5000, random_state=42).fit(Xtr_s, ytr_sc)
                va_p = sy.inverse_transform(m.predict(Xva_s).reshape(-1,1)).flatten()
                s    = smape(y_d[va_m], va_p)
                if s < bev:
                    bev, be = s, m; en_va = va_p
                    en_te = sy.inverse_transform(m.predict(Xte_s).reshape(-1,1)).flatten()
        models['ElasticNet'] = (be, en_va, en_te)

        # RandomForest
        brf, brfv = None, float('inf')
        for ne in [50, 100]:
            for d in [3, 5]:
                m = RandomForestRegressor(n_estimators=ne, max_depth=d, random_state=42, n_jobs=-1).fit(Xtr_s, y_d[tr_m])
                s = smape(y_d[va_m], m.predict(Xva_s))
                if s < brfv: brfv, brf = s, m
        models['RandomForest'] = (brf, brf.predict(Xva_s), brf.predict(Xte_s))

        # XGBoost
        bxg, bxgv = None, float('inf')
        for ne in [50, 100]:
            for d in [3, 4]:
                m = xgb.XGBRegressor(n_estimators=ne, max_depth=d, learning_rate=0.05, random_state=42, n_jobs=-1).fit(Xtr_s, y_d[tr_m])
                s = smape(y_d[va_m], m.predict(Xva_s))
                if s < bxgv: bxgv, bxg = s, m
        models['XGBoost'] = (bxg, bxg.predict(Xva_s), bxg.predict(Xte_s))

        # LightGBM
        blg, blgv = None, float('inf')
        for ne in [50, 100]:
            for d in [3, 4]:
                m = lgb.LGBMRegressor(n_estimators=ne, max_depth=d, learning_rate=0.05, random_state=42, verbose=-1, n_jobs=-1).fit(Xtr_s, y_d[tr_m])
                s = smape(y_d[va_m], m.predict(Xva_s))
                if s < blgv: blgv, blg = s, m
        models['LightGBM'] = (blg, blg.predict(Xva_s), blg.predict(Xte_s))

        winner  = min(models, key=lambda k: smape(y_d[va_m], models[k][1]))
        w_va_p  = models[winner][1]
        pred_te = models[winner][2]

        val_resids = y_d[va_m] - w_va_p
        p10, p90   = np.percentile(val_resids, 10), np.percentile(val_resids, 90)
        pct_p      = pred_te / (np.abs(y_base[te_m]) + 1e-8)
        buy_m  = (pred_te > max(0.0, p90)) & (pct_p >  0.01)
        wait_m = (pred_te < min(0.0, p10)) & (pct_p < -0.01)
        gated  = buy_m | wait_m

        for yt, yp, g in zip(y_d[te_m], pred_te, gated):
            records.append({'fold': fold_id, 'y_true': yt, 'y_pred': yp, 'gated': g})

    all_pair_records[pair_key] = records

print("All predictions collected successfully.")


In [ ]:
n_pairs = len(EVAL_PAIRS)
fig, axes = plt.subplots(n_pairs, 2, figsize=(12, 3.8 * n_pairs))
fig.suptitle("FICOS Walk-Forward Directional Confusion Matrices\n(Aggregated 5 Folds Out-Of-Sample)",
             fontsize=14, fontweight='bold', y=1.01)

pair_summary = []

for row_i, (asset, h) in enumerate(EVAL_PAIRS):
    pair_key = f"{asset}_{h}d"
    recs     = all_pair_records[pair_key]
    df_r     = pd.DataFrame(recs)

    # Clean non-zero moves
    df_r = df_r[(df_r['y_true'] != 0) & (df_r['y_pred'] != 0)].copy()
    df_r['dir_true'] = (df_r['y_true'] > 0).astype(int)
    df_r['dir_pred'] = (df_r['y_pred'] > 0).astype(int)

    # 1. Ungated Confusion Matrix
    cm_ug = confusion_matrix(df_r['dir_true'], df_r['dir_pred'], labels=[1, 0])
    tp_ug, fn_ug = cm_ug[0,0], cm_ug[0,1]
    fp_ug, tn_ug = cm_ug[1,0], cm_ug[1,1]
    acc_ug  = (tp_ug + tn_ug) / (cm_ug.sum() + 1e-8) * 100
    prec_ug = tp_ug / (tp_ug + fp_ug + 1e-8) * 100
    rec_ug  = tp_ug / (tp_ug + fn_ug + 1e-8) * 100

    ax_ug = axes[row_i, 0]
    sns.heatmap(cm_ug, annot=True, fmt='d', cmap='Blues', ax=ax_ug,
                linewidths=0.5, linecolor='white', cbar=False,
                annot_kws={'size': 13, 'weight': 'bold'})
    ax_ug.set_title(f"{pair_key.upper()} — Ungated\nAccuracy: {acc_ug:.1f}% | Precision (UP): {prec_ug:.1f}% | N={len(df_r)}",
                    fontsize=10, fontweight='bold')
    ax_ug.set_xlabel('Predicted', fontsize=9)
    ax_ug.set_ylabel('Actual', fontsize=9)
    ax_ug.set_xticklabels(['UP', 'DOWN'], fontsize=9)
    ax_ug.set_yticklabels(['UP', 'DOWN'], fontsize=9, rotation=0)

    # 2. Gated Confusion Matrix
    df_g = df_r[df_r['gated']].copy()
    ax_gt = axes[row_i, 1]

    if len(df_g) >= 5:
        cm_gt = confusion_matrix(df_g['dir_true'], df_g['dir_pred'], labels=[1, 0])
        tp_gt, fn_gt = cm_gt[0,0], cm_gt[0,1]
        fp_gt, tn_gt = cm_gt[1,0], cm_gt[1,1]
        acc_gt  = (tp_gt + tn_gt) / (cm_gt.sum() + 1e-8) * 100
        prec_gt = tp_gt / (tp_gt + fp_gt + 1e-8) * 100
        rec_gt  = tp_gt / (tp_gt + fn_gt + 1e-8) * 100

        sns.heatmap(cm_gt, annot=True, fmt='d', cmap='Greens', ax=ax_gt,
                    linewidths=0.5, linecolor='white', cbar=False,
                    annot_kws={'size': 13, 'weight': 'bold'})
        ax_gt.set_title(f"{pair_key.upper()} — Gated (P10/P90 Gate)\nAccuracy: {acc_gt:.1f}% | Precision (UP): {prec_gt:.1f}% | N={len(df_g)}",
                        fontsize=10, fontweight='bold')
        pair_summary.append({
            'pair': pair_key,
            'ungated_acc': round(acc_ug, 1),
            'ungated_prec': round(prec_ug, 1),
            'ungated_N': len(df_r),
            'gated_acc': round(acc_gt, 1),
            'gated_prec': round(prec_gt, 1),
            'gated_N': len(df_g)
        })
    else:
        ax_gt.text(0.5, 0.5, f"Gated N={len(df_g)}\n(Sparse/No signals)",
                   ha='center', va='center', transform=ax_gt.transAxes, fontsize=11, color='gray')
        ax_gt.set_title(f"{pair_key.upper()} — Gated (P10/P90 Gate)", fontsize=10, fontweight='bold')
        pair_summary.append({
            'pair': pair_key,
            'ungated_acc': round(acc_ug, 1),
            'ungated_prec': round(prec_ug, 1),
            'ungated_N': len(df_r),
            'gated_acc': None,
            'gated_prec': None,
            'gated_N': len(df_g)
        })

    ax_gt.set_xlabel('Predicted', fontsize=9)
    ax_gt.set_ylabel('Actual', fontsize=9)
    ax_gt.set_xticklabels(['UP', 'DOWN'], fontsize=9)
    ax_gt.set_yticklabels(['UP', 'DOWN'], fontsize=9, rotation=0)

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

df_summary = pd.DataFrame(pair_summary)
df_summary.to_csv('outputs/confusion_matrix_summary.csv', index=False)

print("\n" + "="*65)
print("DIRECTIONAL CONFUSION MATRIX SUMMARY")
print("="*65)
print(df_summary.to_string(index=False))
